In [1]:
import pandas as pd

df_ipc = pd.read_csv("../data/raw/ipc_raw.csv", parse_dates=["fecha"])
df_uf = pd.read_csv("../data/raw/uf_raw.csv", parse_dates=["fecha"])

print("IPC:", df_ipc.shape, "|", df_ipc["fecha"].min(), "->", df_ipc["fecha"].max())
print("UF:", df_uf.shape, "|", df_uf["fecha"].min(), "->", df_uf["fecha"].max())

IPC: (132, 2) | 2015-01-01 00:00:00 -> 2025-12-01 00:00:00
UF: (4018, 2) | 2015-01-01 00:00:00 -> 2025-12-31 00:00:00


In [2]:
df_uf_indexed = df_uf.set_index("fecha")

#UF diaria a mensual. Tomando la ultima entrada de cada més.
uf_mensual = df_uf_indexed["valor"].resample("ME").last()

uf_mensual = uf_mensual.reset_index()
uf_mensual.columns = ["fecha", "uf_valor"]

print(uf_mensual.shape)
uf_mensual.head()

(132, 2)


,fecha,uf_valor
0,2015-01-31,24557.15
1,2015-02-28,24545.23
2,2015-03-31,24622.78
3,2015-04-30,24754.77
4,2015-05-31,24904.75


In [3]:
#Colapsamos a periodo mensual para unir por mes, no por dia.
df_ipc["mes"] = df_ipc["fecha"].dt.to_period("M")
uf_mensual["mes"] = uf_mensual["fecha"].dt.to_period("M")

df_final = df_ipc.merge(uf_mensual, on="mes", how="inner", suffixes=("_ipc", "_uf"))
df_final = df_final.rename(columns={"valor": "ipc_valor"})
df_final = df_final[["mes", "ipc_valor", "uf_valor"]]

print(df_final.shape)
df_final.head()

(132, 3)


,mes,ipc_valor,uf_valor
0,2015-01,68.135885,24557.15
1,2015-02,68.375560,24545.23
2,2015-03,68.805504,24622.78
3,2015-04,69.201029,24754.77
4,2015-05,69.323289,24904.75


In [4]:
base_ipc = df_final["ipc_valor"].iloc[0]
base_uf = df_final["uf_valor"].iloc[0]

df_final["indice_ipc"] = (df_final["ipc_valor"] / base_ipc) * 100
df_final["indice_uf"] = (df_final["uf_valor"] / base_uf) * 100

print(df_final[["mes", "indice_ipc", "indice_uf"]].tail())


         mes  indice_ipc   indice_uf
127  2025-08  159.477994  160.373130
128  2025-09  160.184499  160.790849
129  2025-10  160.251805  161.247010
130  2025-11  160.669339  161.434002
131  2025-12  160.360475  161.777568


"En 11 años, el peso chileno perdió cerca de 38% de su poder de compra frente a los precios — pero un peso guardado en UF prácticamente no perdió nada."

In [5]:
df_final.to_csv("../data/processed/inflacion_procesada.csv", index=False)
print("Guardado:", df_final.shape)

Guardado: (132, 5)
